### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="south_africa_coronary_heart_disease",
    dataset_year="1983",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Kaggle", # Originally from http://statweb.stanford.edu/~tibs/ElemStatLearn/data.html
    original_dataset_source_download_link="https://www.kaggle.com/datasets/waalbannyantudre/south-african-heart-disease-dataset",
    download_description="""
We get the data from Kaggle, as it has the best curation and documentation.

kaggle datasets download waalbannyantudre/south-african-heart-disease-dataset && unzip south-african-heart-disease-dataset.zip && rm south-african-heart-disease-dataset.zip
mkdir -p local-data-warehouse/south_africa_coronary_heart_disease && mv SAHeart.csv local-data-warehouse/south_africa_coronary_heart_disease/
""",
    # References
    academic_reference_bibtex=r"""@article{rossouw1983coronary,
  title={Coronary risk factor screening in three rural communities. The CORIS baseline study.},
  author={Rossouw, JE and Du Plessis, JP and Benad{\'e}, AJ and Jordaan, PC and Kotze, JP and Jooste, PL and Ferreira, JJ},
  journal={South African medical journal= Suid-Afrikaanse tydskrif vir geneeskunde},
  volume={64},
  number={12},
  pages={430--436},
  year={1983}
}
""",
    academic_reference_bibtex_key="rossouw1983coronary",
    license="CC BY-SA 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from Kaggle.

- We remove the row id column.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="chd",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="chd",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "SAHeart.csv")
print("Loaded data shape:", df.shape)

as_cat_type = ["famhist", "chd"]
df[as_cat_type] = df[as_cat_type].astype("category")
df = df.drop(columns=["row.names"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (462, 11)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 462
Columns: 10
Use sampling: False (sample size: 462)
Get row duplicates (staged, merged)...
Using top-9 columns for initial filtering: ['adiposity', 'obesity', 'ldl', 'alcohol', 'tobacco', 'sbp', 'typea', 'age', 'famhist']
Rows remaining as candidates after top-9 filter: 0 (of 462)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,sbp,tobacco,ldl,adiposity,famhist,typea,obesity,alcohol,age,chd
0,143,5.04,4.86,23.59,Absent,58,24.69,18.72,42,0
1,136,2.52,3.95,25.63,Absent,51,21.86,0.00,45,1
2,136,11.20,5.81,31.85,Present,75,27.68,22.94,58,1
3,144,2.40,8.13,35.61,Absent,46,27.38,13.37,60,0
4,153,7.80,3.96,25.73,Absent,54,25.91,27.03,45,0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,famhist,category,0.0,0.0,2.0,"Absent, Present"
1,chd,category,0.0,0.0,2.0,"0, 1"
2,tobacco,float64,0.0,0.0,214.0,"0.0, 6.0, 3.0, 4.0, 0.4, 4.2, 4.5, 0.6, 2.0, 1.2"
3,ldl,float64,0.0,0.0,329.0,"3.95, 4.37, 3.57, 3.3, 4.16, 3.58, 2.4, 5.9, 2.44, 3.79"
4,adiposity,float64,0.0,0.0,408.0,"27.55, 30.79, 21.1, 29.3, 35.95, 24.65, 30.84, 37.83, 23.07, 29.18"
5,obesity,float64,0.0,0.0,400.0,"26.09, 24.86, 22.01, 21.94, 22.51, 24.7, 24.98, 22.59, 27.29, 28.4"
6,alcohol,float64,0.0,0.0,249.0,"0.0, 2.06, 0.51, 8.33, 43.2, 14.4, 11.11, 8.23, 1.03, 3.81"
7,sbp,int64,0.0,0.0,62.0,"136, 134, 128, 132, 124, 118, 126, 130, 138, 122"
8,typea,int64,0.0,0.0,54.0,"52, 57, 50, 54, 49, 56, 60, 61, 55, 47"
9,age,int64,0.0,0.0,49.0,"16, 17, 58, 55, 61, 59, 60, 49, 45, 53"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
sbp,462.0,138.326840,20.496317,101.00,218.00
tobacco,462.0,3.635649,4.593024,0.00,31.20
ldl,462.0,4.740325,2.070909,0.98,15.33
adiposity,462.0,25.406732,7.780699,6.74,42.49
typea,462.0,53.103896,9.817534,13.00,78.00
obesity,462.0,26.044113,4.213680,14.70,46.58
alcohol,462.0,17.044394,24.481059,0.00,147.19
age,462.0,42.816017,14.608956,15.00,64.00


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column  rank                       
chd     1           0    302  65.37
        2           1    160  34.63
famhist 1      Absent    270  58.44
        2     Present    192  41.56

In [8]:
# Target Distribution
target_df

,count,pct
chd,,
0,302,65.37
1,160,34.63


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to south_africa_coronary_heart_disease/019d5dca-3cfd-75a4-9fe7-c820661af84d


019d5dca-3cfd-75a4-9fe7-c820661af84d
9ee4d3f916ac783f75fbc411e263a4b96f5a17fcd06b24f1176709944dfd808a
